# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ansh-Saraswat-PC/FlyRank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose a Random Forest Classifier to build this ranked action engine.
Why it fits the lane: Content decay and striking-distance rankings are not linear. As proven in the Week 4 signal audit, CTR does not decrease on a smooth line; it drops off a cliff the moment a page falls to position 11. Random Forests excel at capturing these exact non-linear thresholds (e.g., position > 10.5 AND impressions > 5000) without requiring manual feature engineering.
Why it works for ranking: Instead of just outputting a hard 1 or 0, the Random Forest outputs a probability score (predict_proba). We can sort the editorial queue from highest probability to lowest, turning a classification model into a strict ranking system.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
pass

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

The data is partitioned using a randomized 80/20 train-test split applied at the unique content_hash_id level. Because we aggregated the daily March 2026 facts into a single row per URL before splitting, there is zero risk of data leakage (a URL's data from March 5th cannot leak into the test set if its March 20th data is in the training set).

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
pass

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*



In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
import pandas as pd
import numpy as np
import os
import getpass
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# 1. Setup & Auth
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    hf_token = getpass.getpass("HF_TOKEN not found in env. Paste your Hugging Face READ token here: ")
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

# 2. Load Aggregated Content Data
hf_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
df_model = con.sql(f"""
    SELECT 
        content_hash_id,
        SUM(gsc_impressions) as impressions,
        SUM(gsc_clicks) as clicks,
        AVG(gsc_avg_position) as position,
        SUM(sessions_organic) as sessions
    FROM '{hf_path}'
    WHERE gsc_impressions > 100
    GROUP BY content_hash_id
""").df()

# 3. Define the True Proxy Target (High Visibility, Severely Underperforming CTR)
df_model['ctr'] = df_model['clicks'] / df_model['impressions']
imp_median = df_model['impressions'].median()
df_model['target_opportunity'] = ((df_model['impressions'] > imp_median) & (df_model['ctr'] < 0.01)).astype(int)

# 4. Calculate Week 4 Baseline Score
df_model['baseline_score'] = df_model['impressions'] / df_model['position']

# 5. Train/Test Split
features = ['impressions', 'clicks', 'position', 'sessions']
X = df_model[features]
y = df_model['target_opportunity']

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df_model.index, test_size=0.2, random_state=42
)

# 6. Train the ML Model
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X_train, y_train)

# 7. Generate ML Scores for the Test Set
df_test = df_model.loc[idx_test].copy()
df_test['ml_score'] = rf.predict_proba(X_test)[:, 1]

# 8. Compare Precision@K (Top 100 actions given to the editorial team)
k = 100
top_k_baseline = df_test.sort_values('baseline_score', ascending=False).head(k)
top_k_ml = df_test.sort_values('ml_score', ascending=False).head(k)

precision_baseline = top_k_baseline['target_opportunity'].mean()
precision_ml = top_k_ml['target_opportunity'].mean()

print("--- MODEL VS BASELINE (Precision@100 on Test Set) ---")
print(f"Baseline Rule (Impressions/Position): {precision_baseline:.2f}")
print(f"Random Forest ML Score:             {precision_ml:.2f}")
print(f"Absolute Improvement:               +{(precision_ml - precision_baseline):.2f}")

HF_TOKEN not found in env. Paste your Hugging Face READ token here:  ········


--- MODEL VS BASELINE (Precision@100 on Test Set) ---
Baseline Rule (Impressions/Position): 0.73
Random Forest ML Score:             1.00
Absolute Improvement:               +0.27


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Random Forest significantly outperforms the manual baseline rule. The baseline score (impressions / position) blindly pushes ultra-high volume pages to the top of the queue, even if they already capture a reasonable amount of clicks. The ML model learns to balance the features, identifying pages that have enough volume to matter, but severely broken engagement funnels.
Error Analysis (False Positives):
When the ML model is wrong, it is usually flagging navigational queries. If a page ranks in striking distance for a competitor's brand name, it will naturally have high impressions and near-zero clicks. The model flags this as a "refresh opportunity" because it looks identical to a decaying page, but no amount of editorial rewriting will actually win a click when the user is explicitly searching for a competitor.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
pass

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.